# Phase 1 — Eval Harness Calibration

Three experiments, in order:

1. **Dataset audit** — is the golden set structurally sound?
2. **No-retrieval baseline** — what does the bare LLM already know?
3. **Judge calibration** — does the LLM judge agree with human labels?

Models per [ADR 0001](../../doc/adr/0001-model-choices.md): generation = Haiku 4.5,
judge = Sonnet 4.6, both on Bedrock. Conclusions graduate to `README.md`,
then to ADRs. This notebook is evidence, not documentation.

In [1]:
import json
import os
from collections import Counter
from pathlib import Path

import boto3
from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent.parent
GOLDEN_PATH = REPO_ROOT / "data" / "golden_dataset.jsonl"
RUBRIC_PATH = REPO_ROOT / "doc" / "judge-rubric.md"
BASELINE_PATH = Path("baseline_answers.json")
HUMAN_LABELS_PATH = Path("human_labels.json")

GENERATION_MODEL = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
JUDGE_MODEL = "global.anthropic.claude-sonnet-4-6"

load_dotenv(REPO_ROOT / ".env")
bedrock = boto3.client("bedrock-runtime", region_name=os.environ["AWS_REGION"])

golden = [json.loads(line) for line in GOLDEN_PATH.read_text().splitlines() if line.strip()]
rubric = RUBRIC_PATH.read_text()
print(f"{len(golden)} golden questions loaded, rubric {len(rubric)} chars")

33 golden questions loaded, rubric 2636 chars


## 1. Dataset audit

Checks: all six types covered, no `TODO` evidence left in non-seed entries,
answerable questions have quoted evidence, unanswerable/out-of-domain have none.

In [2]:
print(Counter(q["type"] for q in golden))

seeds = [q["id"] for q in golden if q.get("status") == "seed_example"]
todos = [q["id"] for q in golden if "TODO" in json.dumps(q) and q.get("status") != "seed_example"]
print(f"seed examples still present: {seeds}")
print(f"non-seed entries with TODOs (must be empty): {todos}")

real = [q for q in golden if q.get("status") != "seed_example"]
print(f"real (non-seed) questions: {len(real)}")

Counter({'factual_single': 24, 'entity_anchored': 3, 'multi_paper': 2, 'unanswerable': 2, 'computable': 1, 'out_of_domain': 1})
seed examples still present: []
non-seed entries with TODOs (must be empty): []
real (non-seed) questions: 33


## 2. No-retrieval baseline

Bare LLM, zero context. For each answerable question: does the model already
know the answer from training data?

**What to look for:** if the model nails a large share of these, the dataset
is skewed toward famous papers and questions must be rewritten toward
recent/obscure work — otherwise retrieval can never prove its value.

In [3]:
def converse(model_id: str, prompt: str, max_tokens: int = 500) -> str:
    response = bedrock.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens},
    )
    return response["output"]["message"]["content"][0]["text"]


def bare_llm_answer(question: str) -> str:
    prompt = (
        "Answer this question about machine learning research concisely. "
        "If you do not know, say so plainly.\n\n" + question
    )
    return converse(GENERATION_MODEL, prompt)


answerable = [q for q in golden if q["expected_behavior"] == "answer" and q.get("status") != "seed_example"]
if not answerable:
    print("No real answerable questions yet — write golden questions first (replace the seeds).")
else:
    baseline = []
    for q in answerable:
        answer = bare_llm_answer(q["question"])
        baseline.append({"id": q["id"], "question": q["question"], "answer": answer})
        print(f"--- {q['id']}: {q['question']}\n{answer}\n")
    BASELINE_PATH.write_text(json.dumps(baseline, indent=2))
    print(f"wrote {len(baseline)} baseline answers to {BASELINE_PATH}")

--- g001: What three retrieval tools does the A-RAG framework provide to the agent?
I don't know the specific details of the A-RAG framework and its three retrieval tools. While there are various retrieval-augmented generation (RAG) frameworks in machine learning research, I'm not certain which framework you're referring to or what its exact three retrieval tools are.

Could you provide more context, such as the paper title or authors, so I can better assist you?



--- g002: What chunk size does A-RAG use when partitioning the corpus for its hierarchical index?
I don't know. I'm not familiar with the specific chunk size used by A-RAG for its hierarchical index partitioning. To get this information, I'd recommend checking the A-RAG paper directly or its official documentation.



--- g003: In A-RAG, what does the chunk-read tool return when the agent tries to read a chunk it has already read?
I don't know the specific behavior of the chunk-read tool in A-RAG when attempting to re-read an already-read chunk. I'm not familiar enough with this particular system's design to answer reliably.

If you have access to the A-RAG paper or documentation, the answer would likely be found in the methods section or tool specification. You might also check if there are implementation details available in any associated code repository.



--- g004: Which four datasets does A-RAG evaluate on?
I don't know which four datasets A-RAG evaluates on. I'm not familiar with a specific paper or method called "A-RAG" with certainty, and I don't want to guess incorrectly about which datasets it uses.

If you could provide more context (such as the authors, publication venue, or a link to the paper), I'd be happy to help you find this information.



--- g005: How many training samples does the EvoTrain-180K dataset actually contain?
I don't know. I'm not familiar with a dataset called "EvoTrain-180K" in machine learning research. It's possible this is a very recent dataset, a specialized/niche dataset, or the name might be slightly different than stated. If you could provide more context or a source, I'd be happy to help.



...

--- g029: What four actions can the ChartWalker agent choose from at each turn?
I don't know what the four actions are that the ChartWalker agent can choose from. I'm not familiar with this specific agent or system. Could you provide more context about ChartWalker or point me to the relevant paper or documentation?



--- g030: What fusion formula does APS-RAG use to merge its three retrieval channels, and what smoothing constant does it use?
I don't know the specific fusion formula or smoothing constant that APS-RAG uses. While I'm familiar with retrieval-augmented generation (RAG) systems generally, I don't have reliable information about the particular implementation details of APS-RAG's multi-channel retrieval fusion mechanism in my training data.

To find this information, I'd recommend checking the APS-RAG paper directly or its official documentation.



--- g031: What formula does APS-RAG's corrective evaluator use to score each draft answer?
I don't know the specific formula that APS-RAG's corrective evaluator uses to score draft answers. While APS-RAG (Adaptive Prompt Selection with Retrieval-Augmented Generation) is a research topic in this space, I'm not familiar with the precise scoring mechanism of its corrective evaluator component.

If you're working with this method, I'd recommend checking the original paper or documentation for the exact formula.



--- g032: Which model powers APS-RAG's self-critique gate, and what does it check?
I don't know this. "APS-RAG" doesn't appear to be a widely recognized model or system in my training data. To answer your question accurately, I would need more context about what APS-RAG is or a reference to where this work was published.

If you could provide more details or a citation, I'd be happy to help.



--- g033: How many nodes and edges does APS-RAG's troubleshooting knowledge graph contain?
I don't know. I'm not familiar with a system called "APS-RAG" or its specific knowledge graph specifications. This appears to be a specialized system that I don't have reliable information about. 

If you're looking for this information, I'd recommend checking the original paper or documentation for APS-RAG directly.

wrote 29 baseline answers to baseline_answers.json


## 3. Human labels

Hand-score the baseline answers against the rubric BEFORE running the judge —
these labels are the ground truth the judge is calibrated against.

The next cell writes a template `human_labels.json` from the baseline answers.
Fill in each `verdict` (`CORRECT` / `WRONG` / `ABSTAINED`) and a short
`reason` by hand, applying `doc/judge-rubric.md` yourself.

In [4]:
if BASELINE_PATH.exists() and not HUMAN_LABELS_PATH.exists():
    template = [
        {"id": row["id"], "verdict": "FILL_ME", "reason": "FILL_ME"}
        for row in json.loads(BASELINE_PATH.read_text())
    ]
    HUMAN_LABELS_PATH.write_text(json.dumps(template, indent=2))
    print(f"wrote template {HUMAN_LABELS_PATH} — fill in verdicts by hand")
elif HUMAN_LABELS_PATH.exists():
    labels = json.loads(HUMAN_LABELS_PATH.read_text())
    unfilled = [l["id"] for l in labels if l["verdict"] == "FILL_ME"]
    print(f"{len(labels)} labels, {len(unfilled)} still unfilled: {unfilled}")
else:
    print("Run the baseline first.")

wrote template human_labels.json — fill in verdicts by hand


## 4. Judge run + disagreement analysis

Runs the judge (rubric as prompt) on the same (question, answer) pairs, then
compares against your labels. Every disagreement forces either a rubric fix
(bump version in `doc/judge-rubric.md`, re-run) or a label fix.
Iterate until agreement ≥ 0.90.

In [5]:
def judge(question: str, answer: str, reference: str | None, evidence: list) -> dict:
    parts = [e["quote"] for e in evidence] + [e["formula"] for e in evidence if "formula" in e]
    evidence_text = "\n".join(parts) or "(none)"
    prompt = f"""You are grading an answer to a research question. Apply these rules exactly:

{rubric}

QUESTION: {question}
REFERENCE ANSWER: {reference}
SUPPORTING EVIDENCE: {evidence_text}
CANDIDATE ANSWER: {answer}

Respond with only a JSON object: {{"verdict": "CORRECT" | "WRONG" | "ABSTAINED", "reason": "<one sentence naming the deciding rule>"}}"""
    raw = converse(JUDGE_MODEL, prompt, max_tokens=200)
    start, end = raw.find("{"), raw.rfind("}") + 1
    return json.loads(raw[start:end])


# Smoke test: one synthetic pair proves the wiring end to end
smoke = judge(
    question="What rank r does the LoRA paper use in its GPT-3 experiments?",
    answer="The paper reports results with r=4.",
    reference="r ranges from 1 to 64 in the experiments; r=4 or 8 works well",
    evidence=[{"quote": "We evaluate r in {1, 2, 4, 8, 64} on GPT-3."}],
)
print("smoke test verdict:", smoke)

smoke test verdict: {'verdict': 'WRONG', 'reason': 'Rule 3 (no partial credit): the candidate answer mentions only r=4 and omits the full range evaluated (r ∈ {1, 2, 4, 8, 64}), making the answer incomplete relative to the reference.'}


In [6]:
if HUMAN_LABELS_PATH.exists() and BASELINE_PATH.exists():
    human = {l["id"]: l for l in json.loads(HUMAN_LABELS_PATH.read_text()) if l["verdict"] != "FILL_ME"}
    by_id = {q["id"]: q for q in golden}
    disagreements = []
    judged = 0
    for row in json.loads(BASELINE_PATH.read_text()):
        if row["id"] not in human:
            continue
        q = by_id[row["id"]]
        verdict = judge(q["question"], row["answer"], q.get("reference_answer"), q.get("evidence", []))
        judged += 1
        if verdict["verdict"] != human[row["id"]]["verdict"]:
            disagreements.append({"id": row["id"], "human": human[row["id"]], "judge": verdict})
    if judged:
        agreement = 1 - len(disagreements) / judged
        print(f"agreement: {agreement:.0%} over {judged} labeled pairs")
        for d in disagreements:
            print(f"\nDISAGREEMENT {d['id']}:\n  human: {d['human']['verdict']} — {d['human']['reason']}\n  judge: {d['judge']['verdict']} — {d['judge']['reason']}")
    else:
        print("No filled labels to judge yet — fill human_labels.json first.")
else:
    print("Need baseline_answers.json and human_labels.json — run sections 2 and 3 first.")

No filled labels to judge yet — fill human_labels.json first.


## 5. Conclusions

> Fill in, then copy to README.md and open ADRs.

- Judge–human agreement: `___%` at rubric `v_`
- No-retrieval baseline accuracy: `___%` (n=`__` answerable questions)
- Questions rewritten as too-famous: `___`
- Rubric changes made and why: `___`